In [1]:
import requests
import json
from sklearn.metrics import classification_report
# Your API token from Open Web UI
token = "sk-1325adfbf63c47f4bf5fd888b144dff6"
API_BASE = "http://194.171.191.227:30080"

# Helper to send API requests
def send_request(data):
    headers = {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json'
    }
    response = requests.post(f"{API_BASE}/api/chat/completions", headers=headers, json=data)
    return response.json()


In [2]:
headers = {
    'Authorization': f'Bearer {token}',
    'Content-Type': 'application/json'
}
response = requests.get(f"{API_BASE}/api/models", headers=headers)
models = response.json()
models


{'data': [{'id': 'llama3.2:3b',
   'name': 'llama3.2:3b',
   'object': 'model',
   'created': 1743407010,
   'owned_by': 'ollama',
   'ollama': {'name': 'llama3.2:3b',
    'model': 'llama3.2:3b',
    'modified_at': '2025-02-27T16:41:08.841805321Z',
    'size': 2019393189,
    'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72',
    'details': {'parent_model': '',
     'format': 'gguf',
     'family': 'llama',
     'families': ['llama'],
     'parameter_size': '3.2B',
     'quantization_level': 'Q4_K_M'},
    'urls': [0]},
   'info': {'id': 'llama3.2:3b',
    'user_id': '2b6ec3d8-8ef1-452b-9d4e-4305d4d23c42',
    'base_model_id': None,
    'name': 'llama3.2:3b',
    'params': {},
    'meta': {'profile_image_url': '/static/favicon.png',
     'description': None,
     'capabilities': {'vision': True, 'citations': True},
     'suggestion_prompts': None,
     'tags': []},
    'access_control': None,
    'is_active': True,
    'updated_at': 1742221616,
    'created_a

In [3]:
import re

def extract_emotion(response_text):
    """
    Extracts a single core emotion or 'neutral' from a long model response.
    """
    emotions = ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral']
    for emotion in emotions:
        if re.search(rf"\b{emotion}\b", response_text.lower()):
            return emotion
    return "unknown"


def classify_emotion(sentence):
    data = {
        "model": "llama3.2:3b",
        "messages": [
            {
                "role": "user",
                "content": f"Analyze the following sentence and classify it as one of the six core emotions (happiness, sadness, anger, surprise, fear, disgust) or neutral: {sentence}"
            }
        ]
    }
    response = send_request(data)
    try:
        raw_output = response["choices"][0]["message"]["content"]
        cleaned_emotion = extract_emotion(raw_output)
        return cleaned_emotion
    except Exception as e:
        print("Error:", e)
        print(json.dumps(response, indent=2))
        return "error"



In [4]:
def classify_emotion(sentence):
    data = {
        "model": "llama3.3:latest",
        "temperature": 0,
        "messages": [
            {
                "role": "user",
                "content": f"Analyze the following sentence and classify it as one of the six core emotions (happiness, sadness, anger, surprise, fear, disgust) or neutral: {sentence}"
            }
        ]
    }
    response = send_request(data)
    try:
        raw_output = response["choices"][0]["message"]["content"]
        cleaned_emotion = extract_emotion(raw_output)
        return cleaned_emotion
    except Exception as e:
        print("Error:", e)
        print(json.dumps(response, indent=2))
        return "error"


In [5]:
test_sentence = "I can't believe he forgot my birthday again."
prediction = classify_emotion(test_sentence)
print("Sentence:", test_sentence)
print("Predicted Emotion:", prediction)


Sentence: I can't believe he forgot my birthday again.
Predicted Emotion: sadness


# Tests

In [6]:
test_data = [
    {"sentence": "I'm so happy you made it!", "label": "happiness"},
    {"sentence": "This is the worst day of my life.", "label": "sadness"},
    {"sentence": "That was totally unexpected!", "label": "surprise"},
    {"sentence": "Why would you do that to me?", "label": "anger"},
    {"sentence": "I’m feeling sick just thinking about it.", "label": "disgust"},
    {"sentence": "What was that noise in the dark?", "label": "fear"},
    {"sentence": "The sky is blue and the grass is green.", "label": "neutral"},
]


In [7]:
y_true = []
y_pred = []

for item in test_data:
    prediction = classify_emotion(item["sentence"])
    y_true.append(item["label"])
    y_pred.append(prediction)
    print(f"Sentence: {item['sentence']}")
    print(f"True: {item['label']} | Predicted: {prediction}")
    print("------")


Sentence: I'm so happy you made it!
True: happiness | Predicted: happiness
------
Sentence: This is the worst day of my life.
True: sadness | Predicted: sadness
------
Sentence: That was totally unexpected!
True: surprise | Predicted: surprise
------
Sentence: Why would you do that to me?
True: anger | Predicted: sadness
------
Sentence: I’m feeling sick just thinking about it.
True: disgust | Predicted: fear
------
Sentence: What was that noise in the dark?
True: fear | Predicted: surprise
------
Sentence: The sky is blue and the grass is green.
True: neutral | Predicted: happiness
------


# Loading my dataset and evaluating the baseline prompts


In [9]:
import pandas as pd

# Update the path to your actual file
df = pd.read_csv("../Data/cropped_df.csv")

# Preview
df[["text", "main_category"]].head()


,text,main_category
0,['i is feeling specially unfriendly she will a...,anger
1,['ane feel rattling disgruntled with myself'],anger
2,['when one heard that my sis had shouted at my...,anger
3,['i feel like atomic number 53 am variety of b...,anger
4,['i will give proper praise to the amish for b...,anger


In [10]:
# drop missing values first
df = df.dropna(subset=["text", "main_category"])

# Select a small sample
sample_df = df.sample(n=45, random_state=42).reset_index(drop=True)


In [11]:
y_true = []
y_pred = []

for _, row in sample_df.iterrows():
    sentence = row["text"]
    true_label = row["main_category"]
    predicted_label = classify_emotion(sentence)

    y_true.append(true_label.lower())
    y_pred.append(predicted_label.lower())

    print(f"Sentence: {sentence}")
    print(f"True: {true_label} | Predicted: {predicted_label}")
    print("------")



Sentence: ['i find astir being naughty for knocker cancer consciousness']
True: happiness | Predicted: happiness
------
Sentence: ['unity seem to wake up every day recently feeling immensely irritable and i cannot quite form tabu why']
True: anger | Predicted: anger
------
Sentence: ['i decided to lay downcast in my bed but then i started to feel rattling violent like i wanted to punch and kick things except i make non wnat to ache anything']
True: anger | Predicted: anger
------
Sentence: ['i sense withal that this is my least successful look and one that upon thoughtfulness i would change the most']
True: happiness | Predicted: sadness
------
Sentence: ['what i did final time was first boil consume a little brine and then you have salt.']
True: neutral | Predicted: happiness
------
Sentence: ['i experience comparable i m trying to convince the most skeptical disbelieve person in the world that yes i really serve have got bipolar disorder']
True: fear | Predicted: sadness
------
Sente

In [12]:
from sklearn.metrics import classification_report

print("Classification Report:")
print(classification_report(y_true, y_pred, zero_division=0))


Classification Report:
              precision    recall  f1-score   support

       anger       0.62      0.67      0.64        12
     disgust       0.67      0.50      0.57         4
        fear       0.50      0.14      0.22         7
   happiness       0.33      0.67      0.44         6
     neutral       0.50      0.11      0.18         9
     sadness       0.23      0.75      0.35         4
    surprise       0.00      0.00      0.00         3

    accuracy                           0.42        45
   macro avg       0.41      0.41      0.34        45
weighted avg       0.47      0.42      0.38        45



In [13]:
prompt = (
    "Classify the following sentence as one of the core emotions: "
    "happiness, sadness, anger, surprise, fear, disgust, or neutral.\n"
    "Only reply with ONE WORD (just the emotion).\n\n"
    f"Sentence: {sentence}"
)


# Iteration 2

In [14]:
def classify_emotion_v2(sentence):
    """
    Prompt V2: Force single-word output.
    """
    prompt = (
        "Classify the following sentence as one of the core emotions: "
        "happiness, sadness, anger, surprise, fear, disgust, or neutral.\n"
        "Only reply with ONE WORD (just the emotion).\n\n"
        f"Sentence: {sentence}"
    )

    data = {
        "model": "llama3.2:3b",
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    
    response = send_request(data)
    try:
        raw_output = response["choices"][0]["message"]["content"]
        cleaned_emotion = extract_emotion(raw_output)
        return cleaned_emotion
    except Exception as e:
        print("Error:", e)
        print(json.dumps(response, indent=2))
        return "error"


In [15]:
y_true_v2 = []
y_pred_v2 = []

for _, row in sample_df.iterrows():
    sentence = row["text"]
    true_label = row["main_category"]
    predicted_label = classify_emotion_v2(sentence)

    y_true_v2.append(true_label.lower())
    y_pred_v2.append(predicted_label.lower())

    print(f"Sentence: {sentence}")
    print(f"True: {true_label} | Predicted: {predicted_label}")
    print("------")


Sentence: ['i find astir being naughty for knocker cancer consciousness']
True: happiness | Predicted: disgust
------
Sentence: ['unity seem to wake up every day recently feeling immensely irritable and i cannot quite form tabu why']
True: anger | Predicted: unknown
------
Sentence: ['i decided to lay downcast in my bed but then i started to feel rattling violent like i wanted to punch and kick things except i make non wnat to ache anything']
True: anger | Predicted: anger
------
Sentence: ['i sense withal that this is my least successful look and one that upon thoughtfulness i would change the most']
True: happiness | Predicted: surprise
------
Sentence: ['what i did final time was first boil consume a little brine and then you have salt.']
True: neutral | Predicted: disgust
------
Sentence: ['i experience comparable i m trying to convince the most skeptical disbelieve person in the world that yes i really serve have got bipolar disorder']
True: fear | Predicted: disgust
------
Senten

In [16]:
from sklearn.metrics import classification_report

print("Classification Report (Prompt V2 — One Word Output):")
print(classification_report(y_true_v2, y_pred_v2, zero_division=0))


Classification Report (Prompt V2 — One Word Output):
              precision    recall  f1-score   support

       anger       0.40      0.17      0.24        12
     disgust       0.06      0.25      0.10         4
        fear       1.00      0.14      0.25         7
   happiness       0.50      0.17      0.25         6
     neutral       1.00      0.44      0.62         9
     sadness       0.17      0.25      0.20         4
    surprise       0.17      0.33      0.22         3
     unknown       0.00      0.00      0.00         0

    accuracy                           0.24        45
   macro avg       0.41      0.22      0.23        45
weighted avg       0.56      0.24      0.30        45



# Iteration 3

In [17]:
def classify_emotion_v3(sentence):
    """
    Prompt V3: Include emotion definitions + one-word output.
    """
    prompt = (
        "You are an expert emotion classifier. Use the following definitions:\n"
        "- Happiness: joy, satisfaction\n"
        "- Sadness: sorrow, disappointment\n"
        "- Anger: frustration, irritation\n"
        "- Surprise: shock, astonishment\n"
        "- Fear: anxiety, worry\n"
        "- Disgust: repulsion, aversion\n"
        "- Neutral: no strong emotion\n\n"
        "Classify the following sentence using one of the above emotions.\n"
        "Only reply with ONE WORD (just the emotion).\n\n"
        f"Sentence: {sentence}"
    )

    data = {
        "model": "llama3.2:3b",
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = send_request(data)
    try:
        raw_output = response["choices"][0]["message"]["content"]
        cleaned_emotion = extract_emotion(raw_output)
        return cleaned_emotion
    except Exception as e:
        print("Error:", e)
        print(json.dumps(response, indent=2))
        return "error"


In [18]:
y_true_v3 = []
y_pred_v3 = []

for _, row in sample_df.iterrows():
    sentence = row["text"]
    true_label = row["main_category"]
    predicted_label = classify_emotion_v3(sentence)

    y_true_v3.append(true_label.lower())
    y_pred_v3.append(predicted_label.lower())

    print(f"Sentence: {sentence}")
    print(f"True: {true_label} | Predicted: {predicted_label}")
    print("------")


Sentence: ['i find astir being naughty for knocker cancer consciousness']
True: happiness | Predicted: unknown
------
Sentence: ['unity seem to wake up every day recently feeling immensely irritable and i cannot quite form tabu why']
True: anger | Predicted: anger
------
Sentence: ['i decided to lay downcast in my bed but then i started to feel rattling violent like i wanted to punch and kick things except i make non wnat to ache anything']
True: anger | Predicted: anger
------
Sentence: ['i sense withal that this is my least successful look and one that upon thoughtfulness i would change the most']
True: happiness | Predicted: disgust
------
Sentence: ['what i did final time was first boil consume a little brine and then you have salt.']
True: neutral | Predicted: disgust
------
Sentence: ['i experience comparable i m trying to convince the most skeptical disbelieve person in the world that yes i really serve have got bipolar disorder']
True: fear | Predicted: disgust
------
Sentence:

In [19]:
from sklearn.metrics import classification_report

print("Classification Report (Prompt V3 — Definitions):")
print(classification_report(y_true_v3, y_pred_v3, zero_division=0))


Classification Report (Prompt V3 — Definitions):
              precision    recall  f1-score   support

       anger       0.67      0.67      0.67        12
     disgust       0.06      0.25      0.09         4
        fear       0.00      0.00      0.00         7
   happiness       1.00      0.17      0.29         6
     neutral       1.00      0.44      0.62         9
     sadness       0.50      0.25      0.33         4
    surprise       0.17      0.33      0.22         3
     unknown       0.00      0.00      0.00         0

    accuracy                           0.36        45
   macro avg       0.42      0.26      0.28        45
weighted avg       0.57      0.36      0.39        45



# Iteration 4

In [20]:
def classify_emotion_v4(sentence):
    """
    Prompt V4: Few-shot examples + one-word output.
    """
    prompt = (
        "Classify the following sentence into one of these emotions: "
        "happiness, sadness, anger, surprise, fear, disgust, or neutral.\n"
        "Only respond with ONE WORD (just the emotion).\n\n"
        "Example 1:\n"
        "Sentence: I just got a promotion!\n"
        "Emotion: happiness\n\n"
        "Example 2:\n"
        "Sentence: I can’t believe you did that to me.\n"
        "Emotion: anger\n\n"
        "Example 3:\n"
        "Sentence: What was that sound in the dark?\n"
        "Emotion: fear\n\n"
        "Example 4:\n"
        "Sentence: My cat passed away yesterday.\n"
        "Emotion: sadness\n\n"
        "Now classify this sentence:\n"
        f"Sentence: {sentence}\n"
        "Emotion:"
    )

    data = {
        "model": "llama3.2:3b",
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = send_request(data)
    try:
        raw_output = response["choices"][0]["message"]["content"]
        cleaned_emotion = extract_emotion(raw_output)
        return cleaned_emotion
    except Exception as e:
        print("Error:", e)
        print(json.dumps(response, indent=2))
        return "error"


In [21]:
y_true_v4 = []
y_pred_v4 = []

for _, row in sample_df.iterrows():
    sentence = row["text"]
    true_label = row["main_category"]
    predicted_label = classify_emotion_v4(sentence)

    y_true_v4.append(true_label.lower())
    y_pred_v4.append(predicted_label.lower())

    print(f"Sentence: {sentence}")
    print(f"True: {true_label} | Predicted: {predicted_label}")
    print("------")


Sentence: ['i find astir being naughty for knocker cancer consciousness']
True: happiness | Predicted: disgust
------
Sentence: ['unity seem to wake up every day recently feeling immensely irritable and i cannot quite form tabu why']
True: anger | Predicted: disgust
------
Sentence: ['i decided to lay downcast in my bed but then i started to feel rattling violent like i wanted to punch and kick things except i make non wnat to ache anything']
True: anger | Predicted: anger
------
Sentence: ['i sense withal that this is my least successful look and one that upon thoughtfulness i would change the most']
True: happiness | Predicted: disgust
------
Sentence: ['what i did final time was first boil consume a little brine and then you have salt.']
True: neutral | Predicted: disgust
------
Sentence: ['i experience comparable i m trying to convince the most skeptical disbelieve person in the world that yes i really serve have got bipolar disorder']
True: fear | Predicted: disgust
------
Sentenc

In [22]:
print("Classification Report (Prompt V4 — Few-Shot):")
print(classification_report(y_true_v4, y_pred_v4, zero_division=0))


Classification Report (Prompt V4 — Few-Shot):
              precision    recall  f1-score   support

       anger       0.80      0.33      0.47        12
     disgust       0.09      0.50      0.15         4
        fear       0.00      0.00      0.00         7
   happiness       0.00      0.00      0.00         6
     neutral       0.75      0.33      0.46         9
     sadness       0.50      0.25      0.33         4
    surprise       0.17      0.33      0.22         3
     unknown       0.00      0.00      0.00         0

    accuracy                           0.24        45
   macro avg       0.29      0.22      0.21        45
weighted avg       0.43      0.24      0.28        45



# Model V2.1

In [23]:
def classify_emotion_v2_1(sentence):
    """
    Prompt V2.1: Refined one-word output with stricter format and label list.
    """
    prompt = (
        "You are an expert emotion classification assistant.\n\n"
        "Your task is to classify the following sentence using ONLY one of the following labels:\n"
        "happiness, sadness, anger, surprise, fear, disgust, neutral.\n\n"
        "Respond with ONLY one word (just the emotion label). Do not explain or include punctuation.\n\n"
        f"Sentence: {sentence}\n"
        "Emotion:"
    )

    data = {
        "model": "llama3.2:3b",
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = send_request(data)
    try:
        raw_output = response["choices"][0]["message"]["content"]
        cleaned_emotion = extract_emotion(raw_output)
        return cleaned_emotion
    except Exception as e:
        print("Error:", e)
        print(json.dumps(response, indent=2))
        return "error"


In [24]:
y_true_v2_1 = []
y_pred_v2_1 = []

for _, row in sample_df.iterrows():
    sentence = row["text"]
    true_label = row["main_category"]
    predicted_label = classify_emotion_v2_1(sentence)

    y_true_v2_1.append(true_label.lower())
    y_pred_v2_1.append(predicted_label.lower())

    print(f"Sentence: {sentence}")
    print(f"True: {true_label} | Predicted: {predicted_label}")
    print("------")


Sentence: ['i find astir being naughty for knocker cancer consciousness']
True: happiness | Predicted: disgust
------
Sentence: ['unity seem to wake up every day recently feeling immensely irritable and i cannot quite form tabu why']
True: anger | Predicted: anger
------
Sentence: ['i decided to lay downcast in my bed but then i started to feel rattling violent like i wanted to punch and kick things except i make non wnat to ache anything']
True: anger | Predicted: fear
------
Sentence: ['i sense withal that this is my least successful look and one that upon thoughtfulness i would change the most']
True: happiness | Predicted: sadness
------
Sentence: ['what i did final time was first boil consume a little brine and then you have salt.']
True: neutral | Predicted: disgust
------
Sentence: ['i experience comparable i m trying to convince the most skeptical disbelieve person in the world that yes i really serve have got bipolar disorder']
True: fear | Predicted: disgust
------
Sentence: 

In [25]:
print("Classification Report (Prompt V2.1 — Refined):")
print(classification_report(y_true_v2_1, y_pred_v2_1, zero_division=0))


Classification Report (Prompt V2.1 — Refined):
              precision    recall  f1-score   support

       anger       0.57      0.33      0.42        12
     disgust       0.11      0.50      0.18         4
        fear       0.20      0.14      0.17         7
   happiness       1.00      0.33      0.50         6
     neutral       1.00      0.22      0.36         9
     sadness       0.25      0.50      0.33         4
    surprise       0.00      0.00      0.00         3

    accuracy                           0.29        45
   macro avg       0.45      0.29      0.28        45
weighted avg       0.55      0.29      0.32        45

